# Phantasm GPU training

Open this notebook from a Phantasm checkout. Use a Linux x86_64 NVIDIA GPU runtime. Read `docs/TRAINING.md` first; the pinned environment needs a real GPU smoke test on your machine.


In [ ]:
import subprocess
from pathlib import Path

repo = Path.cwd()
assert (repo / "src/phantasm/resources/training.txt").is_file(), (
    "Change directory to your Phantasm checkout"
)
subprocess.run(["uv", "venv", "--python", "3.11", ".venv-training"], check=True)
python = str(repo / ".venv-training/bin/python")
subprocess.run(
    [
        "uv",
        "pip",
        "sync",
        "--python",
        python,
        "--require-hashes",
        "src/phantasm/resources/training.txt",
    ],
    check=True,
)
subprocess.run(["uv", "pip", "install", "--python", python, "--no-deps", "-e", "."], check=True)

## Validate your data

Place your disjoint training and validation files in the checkout or update the paths below. This uses the same packaged command as terminal training.


In [ ]:
command = [
    python,
    "-m",
    "phantasm.cli",
    "train",
    "--dataset",
    "dataset_train_sharegpt.jsonl",
    "--validation-dataset",
    "dataset_val_sharegpt.jsonl",
    "--max-steps",
    "120",
    "--output-dir",
    "phantasm_model",
    "--quant-method",
    "q4_k_m",
]
subprocess.run(command + ["--dry-run"], check=True)

In [ ]:
subprocess.run(command, check=True)

## Inspect the result

Check `phantasm_model/run.json`, adapters, checkpoints, and the exported `phantasm_model/gguf/*.gguf` files. Download the GGUF and run `phantasm chat --model PATH` in your local inference environment.


In [ ]:
import json

print(json.dumps(json.loads(Path("phantasm_model/run.json").read_text()), indent=2))